# MinecraftRL 동일 정책 실험 — Colab 후보

미실행 노트북입니다. GPU 런타임을 직접 연결한 뒤 실행하세요. 유료 자원을 자동 선택하지 않습니다. 로컬 PC 원격 제어가 아니라 Colab 내부 MineRL 실험이며, 클라우드 결과를 GTX 1060 결과로 해석하지 않습니다. 전달 ZIP을 업로드합니다.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one MinecraftRL delivery ZIP"
delivery = Path("/content/MinecraftRL-delivery")
assert not delivery.exists(), "Use a fresh destination; do not overwrite prior work"
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        target = (delivery / name).resolve()
        assert target.is_relative_to(delivery.resolve()), "Unsafe ZIP path"
    archive.extractall(delivery)
root = delivery / "project"
for entry in json.loads((delivery / "delivery-manifest.json").read_text()):
    assert hashlib.sha256((root / entry["path"]).read_bytes()).hexdigest() == entry["sha256"]
print(root)


## Colab 전용 렌더링 의존성
이 셀은 일회용 Colab 머신에만 apt 패키지를 설치합니다. Windows/WSL 설치 스크립트는 시스템 apt를 실행하지 않습니다.

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "xvfb", "xauth", "libgl1-mesa-dri", "libglx-mesa0", "libxrender1", "libxtst6", "libxi6", "libglu1-mesa"], check=True)


## 정책별 별도 환경
먼저 vpt를 실행합니다. 끝난 뒤 POLICY를 steve1로 바꿔 동일 절차를 수행합니다. Gym 메타데이터 충돌과 MCP 빌드 patch는 README의 제한을 그대로 적용합니다.

In [ ]:
POLICY = "vpt"  # then "steve1"
assert POLICY in {"vpt", "steve1"}
lock = json.loads((root / "configs/sources.lock.json").read_text())["sources"]
names = [POLICY, "minerl", "gym"] + (["mineclip", "minedojo"] if POLICY == "steve1" else [])
for name in names:
    dest = root / "external" / name
    if dest.exists():
        actual = subprocess.check_output(["git", "-C", str(dest), "rev-parse", "HEAD"], text=True).strip()
        assert actual == lock[name]["commit"]
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", str(dest)], check=True)
    subprocess.run(["git", "-C", str(dest), "remote", "add", "origin", lock[name]["repository"]], check=True)
    subprocess.run(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", lock[name]["commit"]], check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", "--detach", "FETCH_HEAD"], check=True)
def probe(operation):
    return subprocess.run(["bash", str(root / "scripts/wsl-probe.sh"), str(root), POLICY, operation], check=True)
probe("setup")


In [ ]:
probe("download")
probe("preflight")


In [ ]:
probe("suite")


## 결과 보존
실행 실패 시에도 이 셀을 실행해 traceback/setup 로그를 보존합니다. 런타임 강제 중지 시 해제 보증은 미검증입니다. 중단 요청은 `(root / "results/STOP").touch()`로 만들거나 런타임 인터럽트를 사용하며, 다음 실행 전에 로그를 검토해야 합니다.

In [ ]:
import shutil
archive = shutil.make_archive("/content/MinecraftRL-results-" + POLICY, "zip", root / "results")
files.download(archive)
